In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from google import genai

genai_client = genai.Client()

In [9]:
def llm(prompt):
    response = genai_client.models.generate_content(
        model="gemini-2.5-flash",  # O el modelo que prefieras (e.g., gemini-2.5-pro)
        contents=prompt
    )
    return response.text

In [11]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

That's great you've discovered it! To give you the most accurate answer, I need a little more information about the specific course you're referring to.

Please tell me:

1.  **What is the name of the course?**
2.  **Where did you discover it** (e.g., on a specific university website, Coursera, edX, Udemy, a private platform, etc.)?

Once I have that information, I can help you check its enrollment status, deadlines, and how to join (or if you need to wait for a future session).

In general:
*   **Self-paced courses** often allow you to join at any time.
*   **Cohort-based or live courses** usually have specific start dates and enrollment periods. You might be able to join late, or you might need to wait for the next cohort.


In [12]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [13]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [14]:
answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [ ]:
def rag(question):
    search_results = search(question) #1. search in the KB
    user_prompt = build_prompt(question, search_results) #2. build prompt based on the questions and the results from the search
    return llm(user_prompt) #3. send the prompt to the llm

In [15]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [16]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status() #trigger errors
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [17]:
documents[1000]

{'id': '947bf26d84',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 3. Machine Learning for Classification',
 'question': 'pandas.get_dummies() and DictVectorizer(sparse=False) produce the same type of one-hot encodings:',
 'answer': '<{IMAGE:image_1}>\n\n`DictVectorizer(sparse=True)` produces [CSR](https://en.wikipedia.org/wiki/Sparse_matrix#Compressed_sparse_row_(CSR,_CRS_or_Yale_format)) format, which is both more memory efficient and converges better during fit().\n\nIt stores non-zero values and indices instead of adding a column for each class of each feature, which can result in large numbers of columns (e.g., models of cars).\n\nUsing "sparse" format is slower (around 6-8 minutes for Q6 task - Linear/Ridge Regression) for a high number of classes (like car models) and produces slightly worse results in both Logistic and Linear/Ridge Regression.\n\nIt also generates convergence warnings for Linear/Ridge Regression.'}

In [18]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"], #things that are potentially useful to answer the question
    keyword_fields=["course"] #match
)

index.fit(documents)

In [20]:
question

'I just discovered the course. Can I join now?'

In [ ]:
search_results = index.search(
    question, 
    boost_dict = {"question": 2, "section": 0.5}, #boost the importance of the question field
    filter_dict = {"course": "llm-zoomcamp"}, 
    num_results=5
)

In [ ]:
search_results

In [24]:
def search(question, course = 'llm-zoomcamp'):
    boost_dict = {"question": 2, "section": 0.5}
    filter_dict = {"course": course}
    
    return index.search(
        question, 
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results=5
    )

In [28]:
search_results = search(question)

In [26]:
INSTRUCTIONS = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [44]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [45]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [46]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [47]:
print(build_context(search_results))

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [48]:
prompt = build_prompt(question, search_results)


In [49]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [50]:
response = genai_client.models.generate_content(
    model="gemini-2.5-flash",  # O el modelo que prefieras (e.g., gemini-2.5-pro)
    contents=prompt
)

In [ ]:
print(response.text)

Yes, you can join now and start learning the course material. You don't need to register to begin studying or even submitting homework (while the form is open).

However, if your goal is to receive a certificate:
*   You must submit your Capstone project while submissions are still open for the current "live" cohort.
*   You also need to participate in the peer-review process (reviewing 3 capstones), which only happens when the course is actively running and after the submission form is closed.
*   Certificates are not awarded for self-paced completion outside of a live cohort's submission and peer-review windows.

If you miss the current cohort's deadlines for project submission and peer-review, the next opportunity to earn a certificate would be with the course offered in Summer 2025.


In [56]:
print(response.model_dump_json(indent=2))

{
  "sdk_http_response": {
    "headers": {
      "x-gemini-service-tier": "standard",
      "content-type": "application/json; charset=UTF-8",
      "vary": "Origin, X-Origin, Referer",
      "content-encoding": "gzip",
      "date": "Thu, 11 Jun 2026 02:21:01 GMT",
      "server": "scaffolding on HTTPServer2",
      "x-xss-protection": "0",
      "x-frame-options": "SAMEORIGIN",
      "x-content-type-options": "nosniff",
      "server-timing": "gfet4t7; dur=7158",
      "alt-svc": "h3=\":443\"; ma=2592000,h3-29=\":443\"; ma=2592000",
      "transfer-encoding": "chunked"
    },
    "body": null
  },
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "media_resolution": null,
            "code_execution_result": null,
            "executable_code": null,
            "file_data": null,
            "function_call": null,
            "function_response": null,
            "inline_data": null,
            "text": "Yes, you can join now and start learning 

In [61]:
from google.genai import types

# 1. Define las instrucciones del sistema en la configuración
config = types.GenerateContentConfig(
    system_instruction=INSTRUCTIONS
)

# 2. Modifica el historial para usar 'parts' (y elimina el rol system de aquí)
message_history = [
    types.Content(
        role="user",
        parts=[
            types.Part(text=prompt)  # <--- Aquí estaba el detalle
        ]
    )
]

response = genai_client.models.generate_content(
    model="gemini-2.5-flash",  # O el modelo que prefieras (e.g., gemini-2.5-pro)
    contents=message_history,
    config=config
)

print(response.text)

Yes, you can join now and start learning. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded to those who finish the course with a "live" cohort, as it requires peer-reviewing other projects during the course run.


In [63]:
def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    # 1. Define las instrucciones del sistema en la configuración
    config = types.GenerateContentConfig(
        system_instruction=instructions
    )

    # 2. Modifica el historial para usar 'parts' (y elimina el rol system de aquí)
    message_history = [
        types.Content(
            role="user",
            parts=[
                types.Part(text=user_prompt)  # <--- Aquí estaba el detalle
            ]
        )
    ]

    response = genai_client.models.generate_content(
        model=model,  # O el modelo que prefieras (e.g., gemini-2.5-pro)
        contents=message_history,
        config=config
    )

    return response.text

In [64]:
def rag(query, model="gemini-2.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [65]:
answer = rag(question)
print(answer)

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
